# Tutorial: Start Map Failure Drilldown

Audience:
- You are debugging partial failures in a batch workflow and want item-aware failure data.

Prerequisites:
- Run this notebook from the Hypergraph repo root.
- Install the repo environment with `uv sync --group dev`.

Learning goals:
- Use `start_map()` to get a batch handle immediately.
- Read `batch.failures` as structured `FailureCase` objects.
- Inspect individual child `RunResult`s after the batch finishes.


In [ ]:
from __future__ import annotations

import asyncio

from hypergraph import AsyncRunner, Graph, SyncRunner, node


@node(output_name="doubled")
def double(x: int) -> int:
    return x * 2


@node(output_name="boom")
def fail_after_double(doubled: int) -> int:
    raise ValueError("boom")


@node(output_name="doubled")
async def double_async(x: int) -> int:
    await asyncio.sleep(0.01)
    return x * 2


@node(output_name="boom")
async def fail_after_double_async(doubled: int) -> int:
    raise ValueError("boom")

## Sync start_map

Every failed item produces a `FailureCase` with the map item index and the resolved node inputs.


In [ ]:
graph = Graph([double, fail_after_double])
runner = SyncRunner()

batch = runner.start_map(graph, {"x": [1, 2, 3]}, map_over="x", inspect=True)

In [ ]:
result = batch.result(raise_on_failure=False)

{
    "status": result.status.value,
    "failures": [
        {
            "item_index": case.item_index,
            "node_name": case.node_name,
            "inputs": case.inputs,
        }
        for case in batch.failures
    ],
}

## Drill into a child result

Mapped runs capture inspect data per child item. Today `start_map(..., inspect=True)` does not auto-show a single batch widget, so the clearest workflow is to inspect a specific failed child result after the batch finishes.


In [ ]:
failed_item = result[0]
failure = failed_item.failure
assert failure is not None

{
    "status": failed_item.status.value,
    "failure_node": failure.node_name,
    "double_outputs": failed_item.view()["double"].outputs,
}

## Render the failed child inspect view

This shows the same rich inspect widget for one failed batch item.

In [ ]:
failed_item.inspect()

## Async start_map

The async handle keeps the same shape. Start the batch to show the batch widget, then await the result in the next cell.


In [ ]:
async_graph = Graph([double_async, fail_after_double_async])
async_runner = AsyncRunner()

async_batch = async_runner.start_map(async_graph, {"x": [1, 2]}, map_over="x", inspect=True)

In [ ]:
async_result = await async_batch.result(raise_on_failure=False)

{
    "status": async_result.status.value,
    "failures": [
        {
            "item_index": case.item_index,
            "node_name": case.node_name,
            "inputs": case.inputs,
        }
        for case in async_batch.failures
    ],
}

## Render an async failed child inspect view

In [ ]:
async_result[0].inspect()